## Logging into hugging face to load the llama 3.1 8B model for finetunning

In [ ]:
from huggingface_hub import login
login()

from google.colab import drive
drive.mount('/content/drive')

# Set your save path
save_path = "/content/drive/MyDrive/movielens_splits/"
import os
os.makedirs(save_path, exist_ok=True)

Mounted at /content/drive


## Downloading the Movie Lens 1M Dataset

In [ ]:
#Downloading the movie lens datasete
!wget https://files.grouplens.org/datasets/movielens/ml-1m.zip
!unzip ml-1m.zip
!ls

# Downloading the lamma 3.1 8b model
# Installing the dependencies
!pip install -U transformers accelerate bitsandbytes sentencepiece huggingface_hub
!pip install -q peft trl datasets
from huggingface_hub import login
# model_id = "meta-llama/Llama-3.1-8B"

# tokenizer = AutoTokenizer.from_pretrained(model_id)

# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     device_map="auto",
#     load_in_4bit=True,
# )

## Extracting the details from the movie lens dataset for fine tunning the lamma 3.1 model

In [ ]:
import pandas as pd

ratings = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"]
)

movies = pd.read_csv(
    "ml-1m/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["MovieID", "Title", "Genres"]
)

users = pd.read_csv(
    "ml-1m/users.dat",
    sep="::",
    engine="python",
    names=["UserID", "Gender", "Age", "Occupation", "Zip-code"]
)
df = ratings.merge(movies, on="MovieID")
print(df.columns.tolist())

print(ratings.head())
print(movies.head())
print(users.head())

['UserID', 'MovieID', 'Rating', 'Timestamp', 'Title', 'Genres']
   UserID  MovieID  Rating  Timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291
   MovieID                               Title                        Genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy
   UserID Gender  Age  Occupation Zip-code
0       1      F    1          10    48067
1       2      M   56          16    70072
2       3      M   25          15    55117
3       4      M   45           7    02460
4       5      M   25          

## Extracting the features from the dataset for finetunning the llm archietecture`

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

# Storing the total number of user item ratings in the dataset
user_counts = df.groupby("UserID").size()
# Finding all the eligible users with more than 11 item ratings from the dataset
valid_users = user_counts[user_counts >= 11].index
users = df[df["UserID"].isin(valid_users)].reset_index(drop=True)

# Storing all the movies ides into sa set
movie_ids = set(df['MovieID'].unique())
# Mapping the movie id sot their infor's using a candidate vector
movie_id_vector = movies.set_index("MovieID")[["Title", "Genres"]].to_dict("index")

print(f"Total users: {df.UserID.nunique():,}")
print(f"Eligible (>= 11): {len(valid_users):,}")
print(f"Filtered out: {df.UserID.nunique() - len(valid_users):,}")

# Getting informaiton of all teh users in the dataset
all_users = users['UserID'].unique()
# Shuffling all the users together for achieving randomness
np.random.shuffle(all_users)

# Splitting for the training and validation datasets
train_users = all_users[:int(0.9 * len(all_users))]
test_users = all_users[int(0.9 * len(all_users)):]

# Extracting the dataframes for the training and validaton users
train_user_df = users[users['UserID'].isin(train_users)].reset_index(drop=True)
test_user_df = users[users['UserID'].isin(test_users)].reset_index(drop=True)

print(f"\nTrain users: {len(train_users):,}")
print(f"Test users: {len(test_users):,}")
print(f"Train ratings: {len(train_user_df):,}")
print(f"Test ratings : {len(test_user_df):,}")

train_entries = []
val_entries = []

minimum_entries = 11
per_user_val_entries = 1

for user_id, row_data in train_user_df.groupby("UserID"):
    # Sorting according to the users Timestamps before splitting
    row_data = row_data.sort_values("Timestamp").reset_index(drop=True)

    # Skip users with too few interactions
    if len(row_data) < minimum_entries:
        continue

    # Appending the train and validation dataset
    train_entries.append(
        row_data[:-per_user_val_entries]
    )
    val_entries.append(
        row_data[-per_user_val_entries:]
    )

# Converting the arrays into dataframes for loading the rows and storing the csv records
train_df = pd.concat(train_entries).reset_index(drop=True)
val_df = pd.concat(val_entries).reset_index(drop=True)

# Creating a record for all the user's data present in teh train dataset
user_train_history = {}

# Loading and storing each user entry for tracking the history of the user
for user_id, user_data in train_df.groupby("UserID"):
    sorted_history = user_data.sort_values("Timestamp")
    # Convert rows to dictionaries
    user_train_history[user_id] = sorted_history.to_dict("records")

# Saving the training and validation data in local drive
save_path = "./processed_data/"
os.makedirs(save_path, exist_ok=True)
train_df.to_csv(save_path + "train.csv", index=False)
val_df.to_csv(save_path + "val.csv", index=False)

def sample_history_balanced(history_pool, n=10, seed=42):
  """
    Function to find items with distributed ratings to build the train dataset
    for building prompts accordingly
  """
  # Variable to store all the training ouptut pairs
  sampled = []

  for rating_val in [1, 2, 3, 4, 5]:
      bucket = history_pool[history_pool["Rating"] == rating_val]

      if len(bucket) > 0:
        sampled.append(bucket.sample(1, random_state=seed))

  if sampled:
    rating_distributed_results = pd.concat(sampled)
  else:
    rating_distributed_results = pd.DataFrame()

  ## If there are any slots which are left over fill them with other data
  # Remove rows we've already picked
  unused_rows = history_pool[~history_pool.index.isin(rating_distributed_results.index)]

  # Figure out how many more samples we still need
  missing_count = n - len(rating_distributed_results)

  # If we still need rows and there are rows left to use,
  # randomly grab the remaining amount
  if missing_count > 0 and not unused_rows.empty:
      additional_rows = unused_rows.sample(
          n=min(missing_count, len(unused_rows)),
          random_state=seed
      )
      # Add the new rows to the sampled dataframe
      rating_distributed_results = pd.concat([rating_distributed_results, additional_rows])

  return rating_distributed_results.head(n)

def build_train_prompts(feature_df, max_items, user_train, split):
    """Build instruction-style prompt-completion pairs for fine-tuning."""
    examples, missing = [], 0

    for _, row in feature_df.iterrows():
        history = user_train.get(row["UserID"], [])
        if not history:
            missing += 1
            continue

        hist_str = "\n".join(
            f"- {r['Title']} ({r['Genres']}): {r['Rating']}/5"
            for r in history[-max_items:]
        )

        examples.append({
            "prompt": (
                f"A user has rated the following movies:\n{hist_str}\n\n"
                f"Based on these preferences, predict the user's rating (1-5) for:\n"
                f"{row['Title']} ({row['Genres']})"
            ),
            "completion": f"{int(row['Rating'])}",
            "UserID":    int(row["UserID"]),
            "MovieID":   int(row["MovieID"]),
            "rating":    float(row["Rating"]),
            "Timestamp": int(row["Timestamp"]),
        })

    if missing:
        print(f"[{split}] Skipped {missing} rows (no training history)")
    return examples

HISTORY_LEN = 10
train_examples= build_train_prompts(train_df, HISTORY_LEN, user_train_history, "train")
val_examples= build_train_prompts(val_df,   HISTORY_LEN, user_train_history, "val")

print(f"\nTrain prompts : {len(train_examples):,}")
print(f"Val prompts   : {len(val_examples):,}")

def build_test_prompts(test_user_df, movie_id_to_info, all_movie_ids,
                       n_history=10, n_negatives=9, seed=43):
    """
    Function to build the ranking prompts for held-out test users.
    Each prompt has:
      - 10 seen items as history context
      - 10 shuffled candidates (1 true positive + 9 negatives)
    """
    rng = random.Random(seed)

    # Variables to store the rows of users for training(finetunning the llm) and testing
    test_examples  = []
    test_rows      = []

    for user_id, group in test_user_df.groupby("UserID"):
        group = group.sort_values("Timestamp").reset_index(drop=True)
        if len(group) < n_history + 1:
            continue

        # Extracting the last element from the sorted Timestamps for getting the true positive label
        true_positive = group.iloc[-1]

        # Building the dataset by seperating the test (true positive)
        history_pool = group.iloc[:-1]

        # Sample 10 history items with balanced rating distribution
        history_items = sample_history_balanced(history_pool, n=n_history, seed=seed)

        # Extracting all the movies that the users did not see (false positives)
        user_seen_ids= set(group["MovieID"].tolist())
        candidate_pool= list(all_movie_ids - user_seen_ids)
        negatives = rng.sample(candidate_pool, min(n_negatives, len(candidate_pool)))

        # Build candidate list with 1 true positive + 9 negatives, then shuffle
        for i in range(0, len(negatives)):
          negatives[i] = int(negatives[i])
        candidates = [int(true_positive["MovieID"])] + negatives
        rng.shuffle(candidates)
        true_pos_position = candidates.index(true_positive["MovieID"])

        # Format history string
        lines = []
        # Iterrate though all the items present in the extracted history string
        for _, r in history_items.iterrows():
            lines.append(f"- {r.Title} ({r.Genres}): {r.Rating}/5")
        # Build the str for composing a prompt for the model
        hist_str = "\n".join(lines)

        # Making teh candidate list accordingly
        candidate_strs = []
        for i, mid in enumerate(candidates):
            info = movie_id_to_info.get(mid, {})
            Title  = info.get("Title",  f"Movie {mid}")
            Genres = info.get("Genres", "Unknown")
            candidate_strs.append(f"{i+1}. {Title} ({Genres})")
        candidates_str = "\n".join(candidate_strs)

        # Building a prompt for the model to rank the movies
        prompt = (
            f"A user has rated the following movies:\n{hist_str}\n\n"
            f"From the list below, rank which movie this user would most likely enjoy:\n"
            f"{candidates_str}\n\n"
            f"Reply with just the number of the movie they would rate highest."
        )

        # Storing all the prompts for testing the model
        test_examples.append({
            "UserID": int(user_id),
            "prompt": prompt,
            "true_positive_id": int(true_positive["MovieID"]),
            "true_positive_Title": true_positive["Title"],
            "true_positive_pos": true_pos_position,
            "candidates": candidates,
            "true_rating": float(true_positive["Rating"]),
        })

        # Storing all the extracted items in csv file for downstream RAG pipeline
        history_list = history_items[["MovieID", "Title", "Rating"]].to_dict("records")
        row = {"UserID": int(user_id)}
        for i, h in enumerate(history_list):
            row[f"history_{i+1}_MovieID"] = h["MovieID"]
            row[f"history_{i+1}_Title"] = h["Title"]
            row[f"history_{i+1}_rating"] = h["Rating"]
        for i, mid in enumerate(candidates):
            info  = movie_id_to_info.get(mid, {})
            row[f"candidate_{i+1}_MovieID"] = mid
            row[f"candidate_{i+1}_Title"] = info.get("Title", f"Movie {mid}")
            row[f"candidate_{i+1}_isPos"] = (mid == true_positive["MovieID"])
        row["true_positive_MovieID"] = int(true_positive["MovieID"])
        row["true_positive_Title"] = true_positive["Title"]
        row["true_positive_pos"] = true_pos_position
        test_rows.append(row)

    return test_examples, pd.DataFrame(test_rows)

# Creating the prompts for the test examples
test_examples, test_csv_df = build_test_prompts(test_user_df, movie_id_vector, movie_ids)

# Saving all the testing data in local drive
save_path = "/content/drive/MyDrive/movielens_splits/"
os.makedirs(save_path, exist_ok=True)

# Training prompts
with open(save_path + "train_prompts.json", "w") as f:
    json.dump(train_examples, f, indent=2)
with open(save_path + "val_prompts.json", "w") as f:
    json.dump(val_examples, f, indent=2)

# Test ranking data
with open(save_path + "test_ranking_prompts.json", "w") as f:
    json.dump(test_examples, f, indent=2)
test_csv_df.to_csv(save_path + "test_ranking.csv", index=False)

# Raw splits
train_df.to_csv(save_path + "train.csv", index=False)
val_df.to_csv(save_path + "val.csv", index=False)

Total users: 6,040
Eligible (>= 11): 6,040
Filtered out: 0

Train users: 5,436
Test users: 604
Train ratings: 898,964
Test ratings : 101,245

Train prompts : 893,528
Val prompts   : 5,436


In [ ]:
!pip install -q unsloth
import os
os.environ['UNSLOTH_RETURN_LOGITS'] = '1'
print(os.environ.get('UNSLOTH_RETURN_LOGITS'))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 158.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
import os
os.environ['UNSLOTH_RETURN_LOGITS'] = '1'

!pip install -q unsloth

In [ ]:
import os
os.environ['UNSLOTH_RETURN_LOGITS'] = '1'

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import torch
import json

from huggingface_hub import login, whoami
login()
print(whoami())

# ── Load data ──
save_path = "/content/drive/MyDrive/movielens_splits/"

with open(save_path + "train_prompts.json", "r") as f:
    train_examples = json.load(f)

with open(save_path + "val_prompts.json", "r") as f:
    val_examples = json.load(f)

print(f"Train: {len(train_examples):,} | Val: {len(val_examples):,}")

# ── Print GPU info ──
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
!nvidia-smi

# ── Load model — full bf16, no quantization (A100 has plenty of VRAM) ──
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=512,
    load_in_4bit=False,
    dtype=torch.bfloat16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ── Prepare datasets (parallel tokenization) ──
def apply_chat_template(example):
    messages = [
        {"role": "user",      "content": example["prompt"]},
        {"role": "assistant", "content": example["completion"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_dataset = Dataset.from_list(train_examples).map(apply_chat_template, num_proc=8)
val_dataset   = Dataset.from_list(val_examples).map(apply_chat_template, num_proc=8)

# Free memory — raw JSON no longer needed
del train_examples, val_examples
import gc
gc.collect()

print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples: {len(val_dataset):,}")

# Training config — A100-80GB, maximize throughput
# 893K samples × 2 epochs = 1,787K samples
# batch 128 × 1 accum = 128 effective → ~13,960 steps
# Target: finish in <2 hours

training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/llama31-movielens-full",
    num_train_epochs=2,
    per_device_train_batch_size=128,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    tf32=True,
    optim="adamw_torch_fused",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=2000,
    save_steps=2000,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    prediction_loss_only=True,
    max_length=512,
    dataset_text_field="text",
    dataloader_num_workers=4,
    report_to="none",
)

# ── Train ──
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    args=training_args,
)

import time
start = time.time()

trainer.train()

elapsed = time.time() - start
print(f"\nTraining took {elapsed/60:.1f} minutes ({elapsed/3600:.2f} hours)")

# ── Save adapter ──
model_save_path = "/content/drive/MyDrive/llama31-1b-movielens-full-final"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Adapter saved to {model_save_path}")

# ── Save merged full model ──
merged_save_path = "/content/drive/MyDrive/llama31-1b-movielens-full-merged"
model.save_pretrained_merged(merged_save_path, tokenizer, save_method="merged_16bit")
print(f"Merged model saved to {merged_save_path}")
print("Done!")

{'type': 'user', 'id': '69e3eef2ffc1fabeb0973d95', 'name': 'programmerdanger', 'fullname': 'Dev', 'email': 'dmrathod@ucdavis.edu', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1780272000, 'isPro': False, 'avatarUrl': '/avatars/163e334eee4569de11973dcf38da434e.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'now', 'role': 'write', 'createdAt': '2026-05-24T00:12:53.388Z'}}}
Train: 893,528 | Val: 5,436

GPU: NVIDIA A100-SXM4-80GB
Sun May 24 17:34:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|        

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

...

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/content/drive/MyDrive/llama31-1b-movielens-full-merged`: 100%|██████████| 1/1 [00:05<00:00,  5.87s/it]


Successfully copied all 1 files from cache to `/content/drive/MyDrive/llama31-1b-movielens-full-merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:10<00:00, 10.94s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/llama31-1b-movielens-full-merged`
Merged model saved to /content/drive/MyDrive/llama31-1b-movielens-full-merged
Done!
